<a href="https://colab.research.google.com/github/RaviduSenavirathna/Sinhala-Word-Recognizer/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Creating and Fine tuning the Model

Setup (use Colab’s TensorFlow)

In [ ]:
# Colab already has TF; just add Gradio + OpenCV for the demo later
%pip -q install gradio==4.44.0 opencv-python-headless==4.10.0.84

import os, json, numpy as np, tensorflow as tf
from pathlib import Path
print("Python:", os.sys.version)
print("TF    :", tf.__version__)
print("NumPy :", np.__version__)


Project paths & hyper-params

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/si_letters_edge_lite")
DATA_DIR    = PROJECT_DIR / "data"
TRAIN_DIR   = DATA_DIR / "train"
VAL_DIR     = DATA_DIR / "val"      # optional but recommended
TEST_DIR    = DATA_DIR / "test"     # optional

for p in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# image & training config
IMG_SIZE   = 128        # MobileNetV3 works fine at 128
BATCH_SIZE = 32
EPOCHS_H1  = 8          # warmup (head only)
EPOCHS_H2  = 20         # fine-tune
BASE_LR    = 3e-4
FT_LR      = 1e-4

# augmentation
ROT_MAX     = 0.05      # ~2.9°
TRANS_FRAC  = 0.05
ZOOM_FRAC   = 0.05
CONTRAST_FR = 0.10
NOISE_STD   = 0.02

print("Root:", PROJECT_DIR)
print("Put images under data/train/<class_name>/*.jpg|png (same names under val/ for validation)")


Upload your images (run multiple times)

In [ ]:
from google.colab import files

# CHANGE this each time, then run again for each class and split
target = TRAIN_DIR / "ක"            # e.g., ga_base, ga_aa, ka_e, ...
target.mkdir(parents=True, exist_ok=True)

print("Upload images for:", target)
up = files.upload()
for name, data in up.items():
    (target / name).write_bytes(data)

print("Now in:", target)

Preprocessing & dataset (MobileNetV3 needs 3-channel)

In [ ]:
# Keras aug built ONCE (avoid new variables in tf.data)
AUG = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(ROT_MAX, fill_mode="constant", fill_value=0.0),
    tf.keras.layers.RandomTranslation(TRANS_FRAC, TRANS_FRAC, fill_mode="constant", fill_value=0.0),
    tf.keras.layers.RandomZoom(ZOOM_FRAC, fill_mode="constant", fill_value=0.0),
    tf.keras.layers.RandomContrast(CONTRAST_FR),
], name="aug")

def _resize_pad(img, target=IMG_SIZE):
    h,w = tf.shape(img)[0], tf.shape(img)[1]
    scale = tf.cast(target, tf.float32)/tf.cast(tf.maximum(h,w), tf.float32)
    nh = tf.cast(tf.round(tf.cast(h,tf.float32)*scale), tf.int32)
    nw = tf.cast(tf.round(tf.cast(w,tf.float32)*scale), tf.int32)
    img = tf.image.resize(img, (nh,nw), method="bilinear")
    img = tf.image.pad_to_bounding_box(img, (target-nh)//2, (target-nw)//2, target, target)
    return img

def _pre_gray_to_rgb(img, training):
    # source images are grayscale; convert to [0,1], enhance contrast, invert strokes
    img = tf.image.convert_image_dtype(img, tf.float32)        # [0,1], 1-ch
    img = tf.image.adjust_contrast(img, 2.0)
    img = 1.0 - img                                            # strokes bright
    m = tf.reduce_mean(img)
    img = tf.where(img > m, img, img*0.5)                      # soft binarize
    if training:
        img = AUG(img, training=True)
        img = tf.clip_by_value(img + tf.random.normal(tf.shape(img), stddev=NOISE_STD), 0.0, 1.0)
    # MobileNetV3 expects 3-channel; tile grayscale → RGB
    img3 = tf.repeat(img, repeats=3, axis=-1)
    return img3

def make_ds(root_dir: Path, training=True):
    if not any(root_dir.glob("*/*")):
        print(f"⚠️ No data in {root_dir}.")
    ds = tf.keras.utils.image_dataset_from_directory(
        root_dir,
        labels='inferred',
        label_mode='categorical',
        color_mode='grayscale',            # we convert to RGB later
        batch_size=None,
        shuffle=training,
        image_size=(IMG_SIZE, IMG_SIZE)    # placeholder; we do our own resize/pad
    )
    classes = ds.class_names
    def _map(x,y):
        x = tf.image.convert_image_dtype(x, tf.float32)
        x = _resize_pad(x, IMG_SIZE)
        x = _pre_gray_to_rgb(x, training)  # -> RGB
        return x, y
    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds, classes

train_ds, class_names = make_ds(TRAIN_DIR, training=True)
val_available = any(VAL_DIR.glob("*/*"))
val_ds = make_ds(VAL_DIR, training=False)[0] if val_available else None

print("Classes:", class_names)
num_classes = len(class_names)
assert num_classes >= 2, "Need at least 2 class folders in train/"

Model = MobileNetV3-Small (ImageNet → fine-tune)

In [ ]:
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as mnv3_pre

def build_edge_lite(num_classes, train_base=False, from_layer=None):
    inp = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3))
    # Apply MobileNetV3 preprocessing (expects RGB in [0,255] or [0,1]—this fn scales appropriately)
    x = tf.keras.layers.Lambda(mnv3_pre, name="mnv3_pre")(inp)

    base = MobileNetV3Small(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights="imagenet",
        pooling="avg"
    )
    base.trainable = train_base
    if train_base and from_layer is not None:
        for i,layer in enumerate(base.layers):
            layer.trainable = i >= from_layer

    x = base(x)
    x = tf.keras.layers.Dropout(0.25)(x)
    out = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inp, out)
    model.compile(optimizer=tf.keras.optimizers.Adam(BASE_LR),
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])
    return model, base

# Phase 1: train head only (base frozen)
model, base = build_edge_lite(num_classes, train_base=False)
model.summary()

Train – Phase 1 (frozen base)

In [ ]:
CKPT_FILE = PROJECT_DIR / "mnv3_letters.keras"
SAVED_DIR = PROJECT_DIR / "mnv3_letters_savedmodel"

cbs = [
    tf.keras.callbacks.ModelCheckpoint(str(CKPT_FILE), save_best_only=True,
        monitor="val_accuracy" if val_available else "accuracy", mode="max"),
    tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True,
        monitor="val_accuracy" if val_available else "accuracy", mode="max"),
]

hist1 = model.fit(
    train_ds,
    validation_data=val_ds if val_available else None,
    epochs=EPOCHS_H1,
    callbacks=cbs,
    verbose=1
)

Train – Phase 2 (fine-tune last blocks)

In [ ]:
# Unfreeze last ~30% layers for fine-tuning
unfreeze_from = int(len(base.layers) * 0.7)
for i,layer in enumerate(base.layers):
    layer.trainable = i >= unfreeze_from

model.compile(optimizer=tf.keras.optimizers.Adam(FT_LR),
              loss="categorical_crossentropy",
              metrics=["accuracy"])

hist2 = model.fit(
    train_ds,
    validation_data=val_ds if val_available else None,
    epochs=EPOCHS_H2,
    callbacks=cbs,
    verbose=1
)

# Export SavedModel too (Keras 3 uses export for SavedModel)
model.export(str(SAVED_DIR))

# Save class names for later use
(PROJECT_DIR/"class_names.json").write_text(json.dumps(class_names, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved:", CKPT_FILE, "and", SAVED_DIR)


Evaluate (confusion matrix)

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

def evaluate_on(ds, name):
    loss, acc = model.evaluate(ds, verbose=0)
    print(f"{name}: loss={loss:.4f}  acc={acc:.4f}")
    y_true, y_pred = [], []
    for xb, yb in ds:
        p = model.predict(xb, verbose=0)
        y_pred.extend(p.argmax(1))
        y_true.extend(yb.numpy().argmax(1))
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    plt.figure(figsize=(1.2*num_classes, 1.2*num_classes))
    plt.imshow(cm, cmap="Blues")
    plt.title(f"{name} confusion")
    plt.xticks(range(num_classes), class_names, rotation=45, ha="right")
    plt.yticks(range(num_classes), class_names)
    for i in range(num_classes):
        for j in range(num_classes):
            plt.text(j, i, cm[i,j], ha="center", va="center")
    plt.tight_layout(); plt.show()

if val_available:
    evaluate_on(val_ds, "Validation")
elif any(TEST_DIR.glob("*/*")):
    test_ds, _ = make_ds(TEST_DIR, training=False)
    evaluate_on(test_ds, "Test")
else:
    print("No val/test split present.")


Single-image predict helper

In [ ]:
def predict_image(path: str):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=1, expand_animations=False)
    img = _resize_pad(img, IMG_SIZE)
    img = _pre_gray_to_rgb(img, training=False)  # -> 3ch
    x = tf.expand_dims(img, 0)
    p = model.predict(x, verbose=0)[0]
    k = int(np.argmax(p))
    return class_names[k], float(p[k]), p

# try one sample
some = next((TRAIN_DIR/class_names[0]).glob("*"))
predict_image(str(some))


Export to TFLite (dynamic-range + full-int8)

In [ ]:
# Dynamic range (fast & simplest)
TFLITE_DR = PROJECT_DIR/"mnv3_letters_dr.tflite"
conv = tf.lite.TFLiteConverter.from_saved_model(str(SAVED_DIR))
conv.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_dr = conv.convert()
TFLITE_DR.write_bytes(tflite_dr)
print("Wrote:", TFLITE_DR)

# Full INT8 (needs representative dataset)
def rep_ds_gen(folder=TRAIN_DIR, n=200):
    # sample a bunch of images from all classes
    import random
    imgs = []
    for cls in class_names:
        imgs += list((folder/cls).glob("*"))
    random.shuffle(imgs); imgs = imgs[:n]
    for p in imgs:
        img = tf.io.read_file(str(p))
        img = tf.image.decode_image(img, channels=1)
        img = _resize_pad(img, IMG_SIZE)
        img = _pre_gray_to_rgb(img, training=False)  # -> 3ch
        img = tf.expand_dims(img, 0)
        yield [tf.cast(img, tf.float32)]

TFLITE_INT8 = PROJECT_DIR/"mnv3_letters_int8.tflite"
conv = tf.lite.TFLiteConverter.from_saved_model(str(SAVED_DIR))
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = lambda: rep_ds_gen(TRAIN_DIR, n=300)
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type  = tf.int8
conv.inference_output_type = tf.int8
tflite_int8 = conv.convert()
TFLITE_INT8.write_bytes(tflite_int8)
print("Wrote:", TFLITE_INT8)


# Testing

1) One-off test on a single image (path you already have)

In [16]:
import json, numpy as np, tensorflow as tf
from pathlib import Path
from PIL import Image
# === set these ===
PROJECT_DIR = Path("/content/si_letters_edge_lite")  # or your path
CKPT_FILE   = PROJECT_DIR/"mnv3_letters.keras"   # or mnv3_letters.keras if you used MNV3
CLASS_FILE  = PROJECT_DIR/"class_names.json"
IMG_SIZE    = 128

# === load model & labels ===
model = tf.keras.models.load_model(str(CKPT_FILE))
class_names = json.loads((CLASS_FILE).read_text())

# === paste your preprocessing from the notebook ===
def _resize_pad(img, target=IMG_SIZE):
    h,w = tf.shape(img)[0], tf.shape(img)[1]
    scale = tf.cast(target, tf.float32)/tf.cast(tf.maximum(h,w), tf.float32)
    nh = tf.cast(tf.round(tf.cast(h,tf.float32)*scale), tf.int32)
    nw = tf.cast(tf.round(tf.cast(w,tf.float32)*scale), tf.int32)
    img = tf.image.resize(img, (nh,nw))
    img = tf.image.pad_to_bounding_box(img, (target-nh)//2, (target-nw)//2, target, target)
    return img

# If you trained TinyCNN (1-channel):
USE_RGB = False
# If you trained MobileNetV3 (grayscale→RGB):
# USE_RGB = True

def _pre(img, training=False):
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.adjust_contrast(img, 2.0)
    img = 1.0 - img
    m = tf.reduce_mean(img)
    img = tf.where(img>m, img, img*0.5)
    if USE_RGB:
        img = tf.repeat(img, repeats=3, axis=-1)  # tile grayscale to RGB
    return img

def predict_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=1, expand_animations=False)  # read as grayscale
    img = _resize_pad(img, IMG_SIZE)
    img = _pre(img, training=False)
    x = tf.expand_dims(img, 0)
    p = model.predict(x, verbose=0)[0]
    k = int(np.argmax(p))
    return class_names[k], float(p[k]), {class_names[i]: float(p[i]) for i in range(len(class_names))}

# === test ===
test_path = str(next((PROJECT_DIR/"data"/"train"/class_names[0]).glob("*")))  # or put your own path
label, conf, probs = predict_image(test_path)
print("Pred:", label, " Conf:", round(conf, 3))
print("Top-3:", sorted(probs.items(), key=lambda kv: kv[1], reverse=True)[:3])
Image.open(test_path)


TypeError: <class 'keras.src.models.functional.Functional'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras.src.models.functional', 'class_name': 'Functional', 'config': {}, 'registered_name': 'Functional', 'build_config': {'input_shape': None}, 'compile_config': {'optimizer': {'module': 'keras.optimizers', 'class_name': 'Adam', 'config': {'name': 'adam', 'learning_rate': 9.999999747378752e-05, 'weight_decay': None, 'clipnorm': None, 'global_clipnorm': None, 'clipvalue': None, 'use_ema': False, 'ema_momentum': 0.99, 'ema_overwrite_frequency': None, 'loss_scale_factor': None, 'gradient_accumulation_steps': None, 'beta_1': 0.9, 'beta_2': 0.999, 'epsilon': 1e-07, 'amsgrad': False}, 'registered_name': None}, 'loss': 'categorical_crossentropy', 'loss_weights': None, 'metrics': ['accuracy'], 'weighted

2) Drag-and-drop test (choose any image from your computer)

In [ ]:
from google.colab import files
uploaded = files.upload()  # choose one or more images
for name in uploaded.keys():
    label, conf, _ = predict_image(name)
    print(f"{name} -> {label} ({conf:.2f})")


3) Batch test a folder (quick sanity sweep)

In [ ]:
from pathlib import Path
import numpy as np

folder = PROJECT_DIR/"data"/"train"/class_names[0]  # change to any class
hits = 0; total = 0
for p in list(folder.glob("*"))[:20]:
    lab, conf, _ = predict_image(str(p))
    ok = (lab == folder.name)
    hits += int(ok); total += 1
    print(f"{p.name:>20} -> {lab:<15} ({conf:.2f}) {'✓' if ok else '✗'}")
print(f"\nFolder accuracy (sample {total}): {hits/total:.2f}")


4) Small visual probe (see prediction on an image you just took)

In [ ]:
import matplotlib.pyplot as plt
path = str(next((PROJECT_DIR/"data"/"train"/class_names[0]).glob("*")))
pred, conf, _ = predict_image(path)
plt.imshow(np.squeeze(tf.image.decode_image(tf.io.read_file(path), channels=1)), cmap="gray")
plt.title(f"{pred} ({conf:.2f})"); plt.axis("off"); plt.show()


5) Test the TFLite file (optional)

In [ ]:
import numpy as np, tensorflow as tf

TFLITE_PATH = PROJECT_DIR/"tiny_cnn_letters.tflite"  # or mnv3_letters_int8.tflite
interpreter = tf.lite.Interpreter(model_path=str(TFLITE_PATH))
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]

def predict_tflite(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=1)
    img = _resize_pad(img, IMG_SIZE)
    img = _pre(img, training=False)
    x = tf.expand_dims(img, 0).numpy()

    # handle int8 models
    if inp['dtype'] == np.int8:
        scale, zero = inp['quantization']
        x = (x/1.0/scale + zero).astype(np.int8)

    interpreter.set_tensor(inp['index'], x)
    interpreter.invoke()
    y = interpreter.get_tensor(out['index'])[0]

    # dequantize output if needed
    if out['dtype'] == np.int8:
        scale, zero = out['quantization']
        y = (y.astype(np.float32) - zero) * scale

    k = int(np.argmax(y))
    return class_names[k], float(y[k]), {class_names[i]: float(y[i]) for i in range(len(class_names))}

# test tflite
label, conf, probs = predict_tflite(test_path)
print("TFLite:", label, conf)
